# Phase 4: LSTM Core Model
## FreshCart AI — Sequential Recommendation Engine

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import json
import os
import matplotlib.pyplot as plt
from google.colab import drive

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/freshcart/data/processed'
MODELS_DIR = '/content/drive/MyDrive/freshcart/saved_models'

# Load sequences (key is 'sequences', not 'arr_0')
train_data = np.load(f'{DATA_DIR}/X_train.npz')
val_data   = np.load(f'{DATA_DIR}/X_val.npz')
test_data  = np.load(f'{DATA_DIR}/X_test.npz')

X_train = train_data['sequences']
X_val   = val_data['sequences']
X_test  = test_data['sequences']

# Load vocabulary
with open(f'{DATA_DIR}/vocab.json', 'r') as f:
    vocab = json.load(f)

VOCAB_SIZE = len(vocab)  # 5001 (5000 products + 1 PAD)
print(f"Train shape: {X_train.shape}")
print(f"Val shape:   {X_val.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"Vocab size:  {VOCAB_SIZE}")

In [ ]:
# Each sequence of length 100:
# X_in = first 99 tokens (context window)
# y    = last token      (next-item target — scalar per user)
# This aligns with leave-one-out evaluation: model sees all but last item,
# predicts the last item.

X_train_in = X_train[:, :-1]   # shape: (8000, 99)
y_train    = X_train[:, -1]    # shape: (8000,)

X_val_in   = X_val[:, :-1]     # shape: (1000, 99)
y_val      = X_val[:, -1]      # shape: (1000,)

X_test_in  = X_test[:, :-1]    # shape: (1000, 99)
y_test     = X_test[:, -1]     # shape: (1000,)

print(f"X_train_in: {X_train_in.shape}, y_train: {y_train.shape}")
print(f"X_val_in:   {X_val_in.shape},   y_val:   {y_val.shape}")
print(f"X_test_in:  {X_test_in.shape},  y_test:  {y_test.shape}")

In [ ]:
def build_model(vocab_size=5001, embed_dim=128, lstm_units=256, dropout_rate=0.3):
    """
    2-layer stacked LSTM recommender.
    Architecture: Embedding -> LSTM -> LSTM -> Dropout -> Dense(softmax)
    mask_zero=True in Embedding propagates padding mask through LSTM layers.
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
            mask_zero=True,          # D-01: mask PAD token (index 0)
            name='item_embedding'
        ),
        tf.keras.layers.LSTM(
            lstm_units,
            return_sequences=True,   # pass full sequence to second LSTM
            name='lstm_layer_1'
        ),
        tf.keras.layers.LSTM(
            lstm_units,
            return_sequences=False,  # output single vector
            name='lstm_layer_2'
        ),
        tf.keras.layers.Dropout(dropout_rate, name='dropout'),
        tf.keras.layers.Dense(
            vocab_size,
            activation='softmax',
            name='output_layer'
        )
    ], name='LSTMRecommender')
    return model

model = build_model()
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Model compiled. Ready for training.")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Ensure output directory exists on Drive
os.makedirs(MODELS_DIR, exist_ok=True)

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=f'{MODELS_DIR}/lstm_model_checkpoint.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

print("Callbacks configured:")
print(f"  EarlyStopping: patience=3, monitor=val_loss, restore_best_weights=True")
print(f"  ModelCheckpoint: save_best_only=True -> {MODELS_DIR}/lstm_model_checkpoint.h5")

In [ ]:
# D-04: batch_size=512, D-02: pass numpy arrays directly (no tf.data.Dataset)
history = model.fit(
    X_train_in,
    y_train,
    validation_data=(X_val_in, y_val),
    batch_size=512,
    epochs=20,
    callbacks=[early_stopping, model_checkpoint],
    verbose=1
)

stopped_epoch = early_stopping.stopped_epoch
print(f"\nTraining complete.")
print(f"Stopped at epoch: {stopped_epoch if stopped_epoch > 0 else '20 (no early stop)'}")
print(f"Best val_loss: {min(history.history['val_loss']):.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training History', fontsize=14, fontweight='bold')

epochs_range = range(1, len(history.history['loss']) + 1)

# Left panel: Loss
ax1.plot(epochs_range, history.history['loss'],     'b-o', label='Train Loss',      markersize=4)
ax1.plot(epochs_range, history.history['val_loss'], 'r-o', label='Val Loss',        markersize=4)
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right panel: Accuracy
ax2.plot(epochs_range, history.history['accuracy'],     'b-o', label='Train Accuracy', markersize=4)
ax2.plot(epochs_range, history.history['val_accuracy'], 'r-o', label='Val Accuracy',   markersize=4)
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()

# Save to Drive
curve_path = f'{MODELS_DIR}/training_curves.png'
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Training curves saved to: {curve_path}")

In [ ]:
def evaluate_model(model, X_test_in, y_test, k_values=[5, 10]):
    """
    Leave-one-out evaluation mirroring the Apriori baseline protocol (Phase 3).
    - X_test_in: shape (N, 99) — context sequences (all but last item)
    - y_test:    shape (N,)   — ground truth last item (scalar per user)
    - Returns dict of HR@K, Precision@K, MRR
    """
    n_users = len(y_test)
    results = {f'HR@{k}': 0 for k in k_values}
    results.update({f'Precision@{k}': 0 for k in k_values})
    results['MRR'] = 0.0

    print(f"Evaluating on {n_users} test users...")

    # Batch predict for efficiency (avoid per-user loop overhead)
    # predictions shape: (N, vocab_size)
    predictions = model.predict(X_test_in, batch_size=512, verbose=1)

    for i in range(n_users):
        gt_item = int(y_test[i])
        user_preds = predictions[i]  # shape: (vocab_size,)

        # Top-max(k_values) indices sorted by score descending
        max_k = max(k_values)
        top_k_indices = np.argsort(user_preds)[::-1][:max_k]

        # MRR: reciprocal rank of first hit
        for rank, idx in enumerate(top_k_indices, start=1):
            if idx == gt_item:
                results['MRR'] += 1.0 / rank
                break

        # HR@K and Precision@K
        for k in k_values:
            top_k = top_k_indices[:k]
            hit = int(gt_item in top_k)
            results[f'HR@{k}']        += hit
            results[f'Precision@{k}'] += hit / k

    # Normalize
    for key in results:
        results[key] = round(results[key] / n_users, 4)

    return results

lstm_metrics = evaluate_model(model, X_test_in, y_test, k_values=[5, 10])
print("\n=== LSTM Evaluation Results ===")
for metric, value in lstm_metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Load actual Apriori baseline results from Phase 3 CSV
import pandas as pd
baseline_df = pd.read_csv(f'{DATA_DIR}/baseline_results.csv')
apriori_metrics = dict(zip(baseline_df['metric'], baseline_df['value']))

print("\n" + "="*55)
print(f"{'Metric':<15} {'Apriori':>12} {'LSTM':>12} {'Improvement':>14}")
print("="*55)
metric_keys = ['HR@5', 'HR@10', 'Precision@5', 'MRR']
for metric in metric_keys:
    apriori_val = apriori_metrics.get(metric, 0)
    lstm_val    = lstm_metrics.get(metric, 0)
    improvement = lstm_val - apriori_val
    sign        = '+' if improvement >= 0 else ''
    print(f"{metric:<15} {apriori_val:>12.4f} {lstm_val:>12.4f} {sign+f'{improvement:.4f}':>14}")
print("="*55)

In [ ]:
model_save_path = f'{MODELS_DIR}/lstm_model.h5'

# Save model weights + architecture
model.save(model_save_path)
print(f"Model saved to: {model_save_path}")

# Verify: reload and confirm it works
loaded_model = tf.keras.models.load_model(model_save_path)

# Sanity check: output shape should be (1, 5001)
dummy_input = np.zeros((1, 99), dtype=np.int32)
test_output = loaded_model.predict(dummy_input, verbose=0)
assert test_output.shape == (1, 5001), f"Expected output shape (1, 5001), got {test_output.shape}"

print(f"Model loaded successfully from: {model_save_path}")
print(f"Output shape verified: {test_output.shape}  ✓")
print(f"\nArtifact ready for Phase 8 backend: saved_models/lstm_model.h5")

In [ ]:
# Validation checkpoint: LSTM must beat the Apriori baseline
APRIORI_HR5 = apriori_metrics['HR@5']  # Loaded from baseline_results.csv
lstm_hr5    = lstm_metrics['HR@5']
improvement = lstm_hr5 - APRIORI_HR5

# Hard assertion — fails loudly if LSTM did not beat the baseline
assert lstm_hr5 > APRIORI_HR5, (
    f"VALIDATION FAILED: LSTM HR@5 ({lstm_hr5:.4f}) did not exceed "
    f"Apriori HR@5 ({APRIORI_HR5:.4f}). "
    f"Check training: did EarlyStopping fire too early? Is vocab_size correct (5001)?"
)

print("=" * 55)
print("  PHASE 4 VALIDATION CHECKPOINT")
print("=" * 55)
print(f"  ✓ LSTM HR@5    : {lstm_hr5:.4f}")
print(f"  ✓ Apriori HR@5 : {APRIORI_HR5:.4f}")
print(f"  ✓ Improvement  : +{improvement:.4f}  ({improvement/APRIORI_HR5*100:.1f}% relative gain)")
print("=" * 55)
print("  ASSERTION PASSED — LSTM beats Apriori baseline.")
print("=" * 55)

## Phase 4 Summary: LSTM Core Model

### Model Architecture

| Layer | Config |
|-------|--------|
| Embedding | `vocab_size=5001, output_dim=128, mask_zero=True` |
| LSTM 1 | `units=256, return_sequences=True` |
| LSTM 2 | `units=256, return_sequences=False` |
| Dropout | `rate=0.3` |
| Dense | `units=5001, activation='softmax'` |

### Training Configuration

| Parameter | Value |
|-----------|-------|
| Optimizer | Adam (lr=0.001) |
| Loss | SparseCategoricalCrossentropy |
| Batch Size | 512 |
| Max Epochs | 20 |
| Early Stopping | patience=3, restore_best_weights=True |
| Data Pipeline | NumPy arrays → model.fit() directly |

### Evaluation Results (Leave-One-Out, 1000 test users)

| Metric | Apriori Baseline | LSTM | Improvement |
|--------|-----------------|------|-------------|
| HR@5 | (from CSV) | (after training) | (computed) |
| HR@10 | (from CSV) | (after training) | (computed) |
| Precision@5 | (from CSV) | (after training) | (computed) |
| MRR | (from CSV) | (after training) | (computed) |

*Actual values filled in after training run on Colab T4 GPU.*

### Key Design Decisions

- **`mask_zero=True`** in Embedding: padding token (index 0) is masked and ignored by LSTM layers — no information leakage from padding
- **Scalar target**: sequences sliced as `X = seq[:, :-1]`, `y = seq[:, -1]` — model predicts only the last item, matching leave-one-out evaluation exactly
- **Direct numpy fit**: avoids tf.data overhead; all data fits in Colab RAM for 10K user subset
- **Batch size 512**: safe for T4 16GB VRAM, keeps epoch time under ~2 minutes

### Artifacts Produced

- `saved_models/lstm_model.h5` — trained model weights (consumed by Phase 8 FastAPI backend)
- `saved_models/training_curves.png` — loss and accuracy curves

### Next Steps

**Phase 5: Time-Aware LSTM** — add temporal context embeddings (hour-of-day, day-of-week, days-since-last-order) concatenated to the LSTM output to demonstrate Innovation 2.